# 02 — SGPO v2 Results & Visualization

**Research:** Early-Stage CKD Detection Using Hybrid Optimization (SGPO v2)

**Team:** Muhammed Abdel-Hamid | Abdul Rahman Al-Bili

**Supervisor:** Dr. Ibrahim | New Mansoura University | CSE015

---

This notebook loads and visualizes the SGPO v2 optimization results from the completed 30-generation run.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

## 1. Load Saved Results

In [ ]:
RESULTS_DIR = '../results'

with open(f'{RESULTS_DIR}/sgpo_v2_results.json') as f:
    sgpo = json.load(f)

with open(f'{RESULTS_DIR}/baseline_rf_results.json') as f:
    baseline = json.load(f)

conv = pd.read_csv(f'{RESULTS_DIR}/convergence_history.csv')

opt = sgpo['optimization_results']
final = sgpo['final_evaluation']

print(f"Best fitness:     {opt['best_fitness']:.5f}")
print(f"Best AUC:         {opt['best_auc']:.4f}")
print(f"Best Sensitivity: {opt['best_sensitivity']:.4f}")
print(f"Features:         {opt['best_n_features']} / 42")
print(f"Selected:         {opt['selected_features']}")
print(f"HP:               {opt['best_hyperparameters']}")
print(f"Runtime:          {opt['total_time_seconds']:.0f}s ({opt['total_time_seconds']/60:.1f} min)")

## 2. Baseline vs SGPO v2 Comparison

In [ ]:
metrics = ['AUC-ROC', 'Sensitivity', 'Accuracy']
baseline_vals = [baseline['cv_auc_mean'], baseline['cv_sensitivity_mean'], baseline['cv_accuracy_mean']]
sgpo_vals = [final['auc_mean'], final['sensitivity_mean'], final['accuracy_mean']]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, baseline_vals, width, label='Baseline RF (42 feat.)', color='#6B7280', alpha=0.85)
ax.bar(x + width/2, sgpo_vals, width, label='SGPO v2 (8 feat.)', color='#2563EB', alpha=0.85)
ax.set_ylim(0.85, 0.98)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Score')
ax.set_title('Baseline vs SGPO v2 Performance', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nAUC change:   {final['auc_mean'] - baseline['cv_auc_mean']:+.4f} ({(final['auc_mean'] - baseline['cv_auc_mean'])/baseline['cv_auc_mean']*100:+.2f}%)")
print(f"Sens change:  {final['sensitivity_mean'] - baseline['cv_sensitivity_mean']:+.4f}")
print(f"Feature reduction: 42 -> {opt['best_n_features']} (-{42 - opt['best_n_features']}, {(42-opt['best_n_features'])/42*100:.0f}% reduction)")

## 3. Convergence Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Fitness
axes[0].plot(conv['generation'], conv['best_fitness'], 'b-o', markersize=3, linewidth=2)
axes[0].set_xlabel('Generation')
axes[0].set_ylabel('Best Fitness')
axes[0].set_title('Fitness Convergence')

# AUC
axes[1].plot(conv['generation'], conv['gen_auc'], 'g-s', markersize=3, linewidth=2)
axes[1].set_xlabel('Generation')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC Over Generations')

# Features
axes[2].plot(conv['generation'], conv['gen_n_features'], 'r-^', markersize=3, linewidth=2)
axes[2].fill_between(conv['generation'], conv['gen_n_features'], alpha=0.15, color='red')
axes[2].axhline(y=42, color='gray', linestyle='--', alpha=0.5, label='Baseline')
axes[2].set_xlabel('Generation')
axes[2].set_ylabel('Features')
axes[2].set_title('Feature Count Reduction')
axes[2].legend()

# Mark spore dispersal
spore_gens = conv[conv['fgo_action'].str.contains('spore', na=False)]['generation']
for g in spore_gens:
    for ax in axes:
        ax.axvline(x=g, color='orange', linestyle=':', alpha=0.7)

plt.suptitle('SGPO v2 Convergence — 30 Generations', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Final Evaluation (10-Fold Outer CV)

In [ ]:
print('Final Evaluation — 10-fold Outer CV (unseen by optimizer)')
print('=' * 50)
print(f"  Accuracy:    {final['accuracy_mean']:.4f} +/- {final['accuracy_std']:.4f}")
print(f"  AUC-ROC:     {final['auc_mean']:.4f} +/- {final['auc_std']:.4f}")
print(f"  Sensitivity: {final['sensitivity_mean']:.4f} +/- {final['sensitivity_std']:.4f}")
print(f"  Specificity: {final['specificity_mean']:.4f} +/- {final['specificity_std']:.4f}")

# Confusion matrix (from saved aggregated values)
cm = np.array([[25296, 3346], [3211, 26022]])
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=['Non-CKD', 'CKD']).plot(ax=ax, cmap='Blues', values_format=',d')
ax.set_title('SGPO v2 Confusion Matrix (Aggregated)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Selected Features Analysis

In [ ]:
features = opt['selected_features']
categories = {
    'creatinine': 'Lab (Kidney)',
    'age': 'Demographic',
    'n_admissions': 'Admission',
    'avg_los_days': 'Admission',
    'ins_UNKNOWN': 'Insurance',
    'marital_SINGLE': 'Marital',
    'marital_UNKNOWN': 'Marital',
    'marital_WIDOWED': 'Marital',
}

feat_df = pd.DataFrame({
    'Feature': features,
    'Category': [categories.get(f, 'Other') for f in features]
})
print(feat_df.to_string(index=False))

# Category breakdown
cat_counts = feat_df['Category'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
cat_counts.plot(kind='bar', color=['#2563EB', '#DC2626', '#059669', '#EA580C'], ax=ax, alpha=0.85)
ax.set_ylabel('Count')
ax.set_title('SGPO v2 Selected Features by Category', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 6. Summary

| Metric | Baseline | SGPO v2 | Change |
|---|---|---|---|
| AUC-ROC | 0.9561 | 0.9537 | -0.0024 (-0.25%) |
| Sensitivity | 0.8904 | 0.8902 | -0.0002 (-0.02%) |
| Features | 42 | 8 | **-34 (-81%)** |

**Key result:** 81% feature reduction with near-zero performance loss. Only 1 lab test (creatinine) required.